In [ ]:
# ============================================================
# CELL 1: LOAD ALL RESULTS.JSON FILES FROM CHECKPOINTS FOLDER
# ============================================================
import os
import json
import glob
import pandas as pd

CHECKPOINTS_DIR = "checkpoints"

results_files = glob.glob(os.path.join(CHECKPOINTS_DIR, "*", "results.json"))
print(f"Found {len(results_files)} results files")

all_results = []
for rf in results_files:
    with open(rf) as f:
        data = json.load(f)
    all_results.append(data)

df = pd.DataFrame(all_results)
df = df.sort_values(by=["test_macro_f1"], ascending=False).reset_index(drop=True)
df

In [ ]:
# ============================================================
# CELL 2: FORMAT / DISPLAY FULL COMPARISON TABLE
# ============================================================
display_cols = [
    "model", "train_mode", "test_accuracy", "test_macro_f1",
    "test_precision", "test_recall", "total_params",
    "trainable_params", "model_size_mb", "inference_ms_per_image"
]
comparison_df = df[display_cols].copy()
comparison_df["test_accuracy"] = comparison_df["test_accuracy"].round(4)
comparison_df["test_macro_f1"] = comparison_df["test_macro_f1"].round(4)
comparison_df["test_precision"] = comparison_df["test_precision"].round(4)
comparison_df["test_recall"] = comparison_df["test_recall"].round(4)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)
print(comparison_df.to_string(index=False))

In [ ]:
# ============================================================
# CELL 3: SAVE COMPARISON TABLE TO CSV
# ============================================================
output_csv = os.path.join(CHECKPOINTS_DIR, "comparison_table.csv")
comparison_df.to_csv(output_csv, index=False)
print(f"Saved comparison table to {output_csv}")

In [ ]:
# ============================================================
# CELL 4: VISUALIZE — ACCURACY vs MODEL SIZE (efficiency tradeoff)
# ============================================================
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 7))
for _, row in comparison_df.iterrows():
    ax.scatter(row["model_size_mb"], row["test_macro_f1"], s=100)
    ax.annotate(f"{row['model']}\n({row['train_mode']})",
                (row["model_size_mb"], row["test_macro_f1"]),
                fontsize=8, xytext=(5, 5), textcoords="offset points")

ax.set_xlabel("Model Size (MB)")
ax.set_ylabel("Test Macro-F1")
ax.set_title("Accuracy vs Model Size Tradeoff")
plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINTS_DIR, "accuracy_vs_size.png"), dpi=300)
plt.show()

In [ ]:
# ============================================================
# CELL 5: VISUALIZE — BAR CHART OF MACRO-F1 ACROSS ALL RUNS
# ============================================================
comparison_df["run_label"] = comparison_df["model"] + " (" + comparison_df["train_mode"] + ")"

plt.figure(figsize=(12, 6))
sorted_df = comparison_df.sort_values("test_macro_f1")
plt.barh(sorted_df["run_label"], sorted_df["test_macro_f1"], color="steelblue")
plt.xlabel("Test Macro-F1")
plt.title("Macro-F1 Comparison Across All Runs")
plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINTS_DIR, "macro_f1_comparison.png"), dpi=300)
plt.show()

In [ ]:
# ============================================================
# CELL 5: VISUALIZE — BAR CHART OF MACRO-F1 ACROSS ALL RUNS
# ============================================================
comparison_df["run_label"] = comparison_df["model"] + " (" + comparison_df["train_mode"] + ")"

plt.figure(figsize=(12, 6))
sorted_df = comparison_df.sort_values("test_macro_f1")
plt.barh(sorted_df["run_label"], sorted_df["test_macro_f1"], color="steelblue")
plt.xlabel("Test Macro-F1")
plt.title("Macro-F1 Comparison Across All Runs")
plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINTS_DIR, "macro_f1_comparison.png"), dpi=300)
plt.show()

In [ ]:
# ============================================================
# CELL 6: BEST MODEL PER TRAIN_MODE AND OVERALL BEST
# ============================================================
best_per_mode = comparison_df.loc[comparison_df.groupby("train_mode")["test_macro_f1"].idxmax()]
print("=== Best model per training mode ===")
print(best_per_mode.to_string(index=False))

best_overall = comparison_df.loc[comparison_df["test_macro_f1"].idxmax()]
print("\n=== Best overall run ===")
print(best_overall)

In [ ]:
# ============================================================
# CELL 7: VISUALIZE SAMPLE IMAGES FROM EACH CLASS
# ============================================================
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = "Medicinal Plant Leaf Health Split Dataset"  # adjust if needed
TRAIN_DIR = os.path.join(DATA_DIR, "train")

class_names = sorted(os.listdir(TRAIN_DIR))
VALID_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

n_classes = len(class_names)
n_cols = 4
n_rows = (n_classes + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes = axes.flatten()

for i, cls in enumerate(class_names):
    cls_dir = Path(TRAIN_DIR) / cls
    images = [f for f in cls_dir.iterdir() if f.suffix.lower() in VALID_EXTS]
    if images:
        img = mpimg.imread(images[0])
        axes[i].imshow(img)
    axes[i].set_title(cls, fontsize=9)
    axes[i].axis("off")

# hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.suptitle("Sample Image From Each Class", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINTS_DIR, "class_samples.png"), dpi=300)
plt.show()

In [ ]:
# ============================================================
# CELL 8: VISUALIZE MISCLASSIFIED IMAGES FOR THE BEST MODEL
# ============================================================
best_run_tag = f"{best_overall['model']}_{best_overall['train_mode']}"
best_run_dir = os.path.join(CHECKPOINTS_DIR, best_run_tag)
pred_log_path = os.path.join(best_run_dir, "test_predictions_log.csv")

assert os.path.exists(pred_log_path), f"No prediction log found at {pred_log_path} — run the test-prediction cell for this model first."

pred_log_df = pd.read_csv(pred_log_path)
misclassified = pred_log_df[~pred_log_df["correct"]]

print(f"Best model: {best_run_tag}")
print(f"Total misclassified: {len(misclassified)} / {len(pred_log_df)}")

N_SHOW = 16  # how many misclassified samples to display
sample_misclass = misclassified.sample(min(N_SHOW, len(misclassified)), random_state=42)

n_cols = 4
n_rows = (len(sample_misclass) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes = axes.flatten()

for i, (_, row) in enumerate(sample_misclass.iterrows()):
    img = mpimg.imread(row["image_path"])
    axes[i].imshow(img)
    axes[i].set_title(f"True: {row['true_class']}\nPred: {row['predicted_class']}\nConf: {row['confidence']:.2f}",
                       fontsize=8, color="red")
    axes[i].axis("off")

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.suptitle(f"Misclassified Test Images — {best_run_tag}", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(best_run_dir, "misclassified_samples.png"), dpi=300)
plt.show()

In [ ]:
# ============================================================
# CELL 9: COMPUTE GFLOPs FOR EACH MODEL (updated with custom_cnn support)
# ============================================================
import torch
import torch.nn as nn
from torchvision import models
from thop import profile

IMG_SIZE = 128
NUM_CLASSES = len(class_names)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- custom CNN architecture (must match your training notebook exactly) ---
class LightCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x


def build_model(name, num_classes):
    if name == "mobilenet_v3_small":
        m = models.mobilenet_v3_small(weights=None)
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, num_classes)
    elif name == "shufflenet_v2":
        m = models.shufflenet_v2_x1_0(weights=None)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=None)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    elif name == "custom_cnn":
        m = LightCNN(num_classes)
    elif name == "wider_cnn":
        m = WiderCNN(num_classes)
    else:
        raise ValueError(f"Unknown model: {name}")
    return m


gflops_results = {}
unique_models = comparison_df["model"].unique()

for model_name in unique_models:
    model = build_model(model_name, NUM_CLASSES).to(DEVICE)
    model.eval()
    dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)

    macs, params = profile(model, inputs=(dummy_input,), verbose=False)
    gflops = (macs * 2) / 1e9

    gflops_results[model_name] = round(gflops, 4)
    print(f"{model_name}: {gflops:.4f} GFLOPs | {params:,} params")

    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

print("\nGFLOPs per model:", gflops_results)

In [ ]:
# ============================================================
# CELL 10: MERGE GFLOPs INTO COMPARISON TABLE
# ============================================================
comparison_df["gflops"] = comparison_df["model"].map(gflops_results)

# reorder columns for clarity
final_cols = [
    "model", "train_mode", "test_accuracy", "test_macro_f1",
    "test_precision", "test_recall", "total_params", "trainable_params",
    "model_size_mb", "gflops", "inference_ms_per_image"
]
comparison_df = comparison_df[final_cols]

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)
print(comparison_df.to_string(index=False))

comparison_df.to_csv(os.path.join(CHECKPOINTS_DIR, "comparison_table.csv"), index=False)
print(f"\nUpdated comparison table saved to {os.path.join(CHECKPOINTS_DIR, 'comparison_table.csv')}")

In [ ]:
# ============================================================
# CELL 11: EFFICIENCY VISUALIZATION — GFLOPs vs ACCURACY
# ============================================================
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 7))
for _, row in comparison_df.iterrows():
    ax.scatter(row["gflops"], row["test_macro_f1"], s=100)
    ax.annotate(f"{row['model']}\n({row['train_mode']})",
                (row["gflops"], row["test_macro_f1"]),
                fontsize=8, xytext=(5, 5), textcoords="offset points")

ax.set_xlabel("GFLOPs")
ax.set_ylabel("Test Macro-F1")
ax.set_title("Computational Cost (GFLOPs) vs Accuracy")
plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINTS_DIR, "gflops_vs_accuracy.png"), dpi=300)
plt.show()